<a href="https://colab.research.google.com/github/jawahirkhaleel/-jawahirkhaleel-/blob/main/Mbabane_urban_heat_islands.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install osmnx;
!pip install pathlib;
!pip install requests;
!pip install time;

ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time


In [ ]:
import geopandas as gdp
import osmnx as ox
import pandas as pd
import numpy as np
from pathlib import Path
import json
import ee
import time
import requests
import rasterio

In [ ]:
#configurations---------------------------------
CITY = "/content/Mbabane.shp"
MAX_BUILDINGS = 1000000
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:

print("Urban heat islands analysis with real Landsat data")
print(f"Study area: {CITY}")
print(f"Buildings: {MAX_BUILDINGS} (all processed with batching)")

# Get city boundary from the local shapefile
print("\n1. Loading city boundary from local storage")

boundary_file = Path(CITY)

if not boundary_file.exists():
    raise FileNotFoundError(
        f"Shapefile not found: {boundary_file}"
    )

city_gdf = gpd.read_file(boundary_file)

if city_gdf.empty:
    raise ValueError("The boundary shapefile contains no features.")

if city_gdf.crs is None:
    raise ValueError(
        "The shapefile has no CRS defined. "
        "Please define its original CRS before continuing."
    )

# Ensure geographic coordinates for compatibility with geographic workflows
city_gdf = city_gdf.to_crs("EPSG:4326")

# Combine all boundary features
city_geom = city_gdf.geometry.union_all()

# Validate geometry type
if city_geom.geom_type not in ("Polygon", "MultiPolygon"):
    raise ValueError(
        f"Expected a polygon boundary, got {city_geom.geom_type}"
    )

print("City boundary loaded successfully.")
print(f"Geometry type: {city_geom.geom_type}")
print(f"CRS: {city_gdf.crs}")
print(f"Bounds: {city_geom.bounds}")
print(f"Area (square degrees): {city_geom.area}")




Urban heat islands analysis with real Landsat data
Study area: /content/Malkerns.shp
Buildings: 1000000 (all processed with batching)

1. Loading city boundary from local storage
City boundary loaded successfully.
Geometry type: Polygon
CRS: EPSG:4326
Bounds: (31.126794251985498, -26.596823908782962, 31.259405791754965, -26.4846168650981)
Area (square degrees): 0.008620167291680095


In [ ]:
#get buildings from OSM
print("\n2. getting buildings from OSM")

#Increase OSM timeout
ox.settings.timeout = 600 #10 minutes
ox.settings.requests_timeout = 600

buildings = ox.features_from_polygon(city_geom, tags={"building":True})
buildings = buildings[buildings.geometry.type.isin(["Polygon", "MultiPolygon"])].head(MAX_BUILDINGS)
print(f"hot{len(buildings)} buildings")


2. getting buildings from OSM
hot3456 buildings


In [ ]:
def authenticate_gee():
  """authenticate with GEE"""
  try:
    ee.Initialize()

    print("GEE Authenticated")
    return True
  except Exception as e:
    print(f"GEE auth failed:{e}")
    return False


In [ ]:

def get_landsat_temperature_gee(city_gdf, buildings_gdf):
    """Get Landsat 8/9 Collection 2 Level 2 data from GEE."""

    if not authenticate_gee():
        raise Exception("GEE authentication required")

    # Ensure the study area uses WGS84
    city_gdf = city_gdf.to_crs("EPSG:4326")

    # Use the actual study-area polygon
    city_geom = city_gdf.geometry.union_all()

    # Convert the polygon to Earth Engine geometry
    region = ee.Geometry(
        city_geom.__geo_interface__
    )

    # Get Landsat 8/9 Collection 2 Level 2
    landsat = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .merge(
            ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
        )
        .filterBounds(region)
        .filterDate("2026-09-15", "2026-10-18")
        .filter(ee.Filter.lt("CLOUD_COVER", 10))
        .sort("CLOUD_COVER")
    )

    # Check the number of matching scenes
    scene_count = landsat.size().getInfo()
    print(f"Matching Landsat scenes: {scene_count}")

    if scene_count == 0:
        raise Exception(
            "No Landsat scenes found for this date range "
            "with cloud cover below 10%. Try a wider date range "
            "or a higher cloud-cover threshold."
        )

    # Select the least-cloudy scene
    image = landsat.first()
    info = image.getInfo()

    print(f"Selected Landsat scene: {info['id']}")

    return image

    #get thermal band
    thermal = landsat.select('ST_B10')

    print("processing all buildings in batches")

    batch_size = 1500
    all_temperature = []
    processed_count = 0


    for i in range (0, len (buildings_gdf), batch_size):
      batch = buildings_gdf.iloc[i:i+batch_size]

      print(f"processing batch {i//batch_size+1}: {len(batch)} buildings")

      #Convert buildings to GEE format
      buildings_coords = []
      for idx, building in batch.iterrows():
        centroid = building.geometry.centroidbuilding_coords.append([centroid.x,centroid.y])

      #create GEE points
      points = ee.featurecollection([
          ee.Feature(ee.Geometry.Point(coords), {'id':j})
          for j, coords in enumerate(buildings_coords)
      ])

      #sample temperature values
      samples = thermal.sampleRegions(
          collection = points,
          scale = 30,
          geometries = True
      )

      #get the results
      samples_info = samples.getinfo()
      features = sample_info['features']

      #extract temperature
      batch_temps = []
      for feature in features:
        temp = feature['properties'].get('ST_B10', None)
        batch_temps.append(temp if temp is not None else np.nan)

      all_temperatures.extend (batch_temps)
      processed_count += len (batch)

      print(f"batch complete: {len(features)} temperature extracted")

      #small delay to respectful GEE
      time.sleep(0.1)

    print(f"toatal processed:{processed_count} buildings")

    #add temperaturesto all buildings
    buildings_gdf = buildings_gdf.copy()
    buildings_gdf['LST_C'] = all_temperatures

    #apply temperature scaling (landsat thermal conversion)
    buildings_gdf['LST_C'] = buildings_gdf['LST_c']*0.00341802 + 149.0 - 273.15

    #remove buildings without temperature data
    valid_buildings = buildings_gdf.dropna(subset=['LST_C'])

    print(f"{len(valid_buildings)} buildings with valid temperature data")
    if len(valid_buildings)>0:
      print(f" temperature range:{valid_buildings['LST_C'].min():.1f}C - {valid_buildings['LST_C'].max():.1f}C")

    return valid_buildings

In [ ]:
#get landsat temperature data from GEE
print("\n3. getting landsat temperature data...")

try:
  buildins = get_landsat_temperature_gee(city_gdf, buildings)
  #add temperature classification for real data
  buildings["lst_bin"] = pd.cut(buildings["LST_c"],
                                bins = [buildings["LST_C"].min(), buildings["LST_c"].quantile(0.25),
                                        buildings["LST_C"].quantile(0.5), buildings["LST_C"].quantile(0.75),
                                        buildings["LST_C"].max()],
                                labels = ["Q1 (coolest)", "Q2", "Q3", "Q4(hottest)"])
  data_type = "REAL Landsat (GEE)"
except Exception as e:
  print(f" GEE failed: {e}")
  print(" falling back to sysnthetic data")
  #Fallback to synthetic data
  np.random.seed(42)
  base_temp = 26.0
  urban_heat_effect = np.random.normal(3,1.5,len(buildings))
  buildings["LST_C"] = base_temp + urban_heat_effect + np.random.normal(0,2, len(buildings))
  buildings["lst_bin"] = pd.cut(buildings["LST_C"],
                                bins = [15,23,29,35,45],
                                labels = ["cool", "moderate", "warm", "hot"])

  data_type = "synthetic (GEE failed)"

#save results with only essential properties
print("\n5. saving results....")

#keeping only essential properties to reduce file size
essential_columns = ['building', 'LST_C', 'lst_bin', 'geometry']
if 'id' in buildings.columns:
  essential_columns.insert(0,'id')

#filter to only essential columns
buildings_clean = buildings[essential_columns].copy()

results_path = OUTPUT_DIR/"urban_heat_islands_Mbabane.geojson"
buildings_clean.to_file(results_path, driver = "GeoJSON")

#show file size reduction
original_size = len(buildings)
print(f"kept only essential properties: {essential_columns}")
print(f"file size optimized for {len(buildings_clean)} buildings")


3. getting landsat temperature data...
GEE auth failed:Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.
 GEE failed: GEE authentication required
 falling back to sysnthetic data

5. saving results....
kept only essential properties: ['building', 'LST_C', 'lst_bin', 'geometry']
file size optimized for 3456 buildings


In [ ]:
summary = {
    "city": CITY,
    "buildings": len(buildings),
    "temp_range": f"{buildings['LST_C'].min():.1f}C - {buildings['LST_C'].max():.1f}C",
    "landsat_status": data_type
}

with open (OUTPUT_DIR/"summary.json", 'w') as f:
  json.dump(summary, f, indent=2)

print("done!")
print(f"results in {OUTPUT_DIR}/")
print(f"buildings: {len(buildings)}")
print(f"temperature: {buildings['LST_C'].min():.1f}C - {buildings['LST_C'].max():.1f}C")


done!
results in results/
buildings: 3456
temperature: 20.0C - 37.9C
